In [ ]:
# 必要なモジュールをインポート
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from typing import Annotated
from typing_extensions import TypedDict
from langchain_community.tools.tavily_search import TavilySearchResults
from langgraph.graph import StateGraph
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition
from langgraph.checkpoint.memory import MemorySaver
from langchain_core.messages import ToolMessage

# ===== Stateクラスの定義 =====
class State(TypedDict):
    messages: Annotated[list, add_messages]

# ===== グラフの構築 =====
def build_graph(model_name):
    # グラフのインスタンスを作成
    graph_builder = StateGraph(State)

    # 検索ツールの定義
    tool = TavilySearchResults(max_results=2)
    tools = [tool]

    # 言語モデルの定義
    llm = ChatOpenAI(model_name=model_name)
    # ツール定義の紐づけ
    llm_with_tools = llm.bind_tools(tools)

    # チャットボットノードの作成
    def chatbot(state: State):
        return {"messages": [llm_with_tools.invoke(state["messages"])]}

    # グラフにチャットボットノードを追加
    graph_builder.add_node("chatbot", chatbot)

    # ツールノードの作成
    tool_node = ToolNode(tools)

    # グラフにツールノードを追加
    graph_builder.add_node("tools", tool_node)

    # 条件付エッジの作成
    graph_builder.add_conditional_edges(
        "chatbot",
        tools_condition, # ツール呼出と判断したらツールノードを呼ぶ
    )

    # ツールが呼び出されるたびに、チャットボットに戻って次のステップを決定
    # ツールからチャットボットへの戻りエッジを作成
    graph_builder.add_edge("tools", "chatbot")

    # 開始ノードの指定
    graph_builder.set_entry_point("chatbot")

    # 記憶を持つ実行可能なステートグラフの作成
    memory = MemorySaver()
    return graph_builder.compile(checkpointer=memory)

# ===== グラフ実行関数 =====
def stream_graph_updates(user_input: str):
    events = graph.stream(
        {"messages": [("user", user_input)]},
        {"configurable": {"thread_id": "1"}},
        stream_mode="values")
    # 結果をストリーミングで得る
    for event in events:
        last_message = event["messages"][-1]
        # 途中経過のJSONは弾く
        if not isinstance(last_message, ToolMessage):
            print(last_message.content, flush=True)

# ===== メイン実行ロジック =====
# 環境変数の読み込み
load_dotenv("../.env")
os.environ['OPENAI_API_KEY'] = os.environ['API_KEY']

# モデル名
MODEL_NAME = "gpt-4o-mini" 

# グラフの作成
graph = build_graph(MODEL_NAME)

# メインループ
while True:
    user_input = input()
    if user_input.strip()=="":
        print("ありがとうございました!")
        break
    stream_graph_updates(user_input)


こんにちは！
こんにちは！今日はどんなことをお手伝いできますか？
1たす2は？
1たす2は3です。何か他にお手伝いできることはありますか？
台湾観光について検索結果を教えて

台湾観光に関する情報をいくつか見つけました。

1. **台湾の観光資源と歴史的背景**:
   - 台湾は四面を海に囲まれ、常夏の気候で、豊かな自然と多様な文化を持つ観光地です。特に、南国植物（椰子、マンゴー、パパイヤなど）が豊富で、観光価値も高いとされています。観光事業の発展には、観光組織の整備や宣伝の強化、観光設備の充実が必要とされています。
   - 詳細は[こちら](https://www.agu.ac.jp/pdf/graduate/thesis/letters1/35-3.pdf)で確認できます。

2. **おすすめ観光スポット**:
   - **九份**: 台北近郊の美しい山間の町で、伝統的な街並みや美しい夜景が楽しめます。
   - **士林夜市**: 台北最大の夜市で、たくさんの屋台やレストランがあります。
   - **日月潭**: 台中に位置する美しい湖で、自然を楽しむことができます。
   - **台北101**: 台湾の象徴的な超高層ビルで、展望台からの眺めは素晴らしいです。
   - **野柳地質公園**: 独特な岩の形状が見られる地質公園です。
   - 詳細は[こちら](https://www.hankyu-travel.com/guide/taiwan/)で確認できます。

台湾は自然と文化が豊かで、観光スポットも多様ですので、訪れる際にはぜひこれらの場所をチェックしてみてください！他に知りたいことがあれば教えてください。
ありがとうございました!
